# 2장. 지점 파일 다섯 개를 한 표로 합치기

지점마다 파일 모양이 조금씩 다르다. 다른 점을 먼저 눈으로 확인하고, 읽는 함수 하나로 맞춘다.

In [1]:
import unicodedata
from pathlib import Path

import pandas as pd

DATA = Path("../data")
WORK = Path("../work")
WEEK = "2026-11-02"

files = sorted((DATA / "raw" / WEEK).glob("*.xlsx"))
len(files)

5

## 1. 파일마다 맨 위 네 줄만 보기
`header=None`으로 읽으면 열 이름 줄도 데이터처럼 그대로 보인다.

In [2]:
for f in files:
    print("▼", f.name)
    top = pd.read_excel(f, header=None, nrows=4)
    print(top.to_string(index=False, header=False))
    print()

▼ 강남점_주간판매.xlsx
       전표번호                 판매일  품목코드        품목명 수량    단가     금액
GN-1102-001 2026-11-02 00:00:00 P-303    컵라면 24개  5 21600 108000
GN-1102-002 2026-11-02 00:00:00 P-403 USB 충전 케이블  2  5900  11800
GN-1102-003 2026-11-02 00:00:00 P-202  종이컵 1000개  4 15400  61600

▼ 광주점_주간판매.xlsx
       전표번호                 판매일  품목코드         품목명 수량    단가     금액
GJ-1102-001 2026-11-02 00:00:00 P-402      멀티탭 4구 10 12500 125000
GJ-1102-002 2026-11-02 00:00:00 P-202   종이컵 1000개  5 15400  77000
GJ-1102-003 2026-11-02 00:00:00 P-101 A4 복사용지 1박스  6 24500 147000

▼ 대전점_주간판매.xlsx
       전표번호                  일자    코드     상품명 판매수량  판매단가   판매금액
DJ-1102-001 2026-11-02 00:00:00 P-203 주방세제 4개   15  8700 130500
DJ-1102-002 2026-11-02 00:00:00 P-201 물티슈 10팩   24 11900 285600
DJ-1102-003 2026-11-02 00:00:00 P-402  멀티탭 4구    3 12500  37500

▼ 부산점_주간판매.xlsx
       전표번호        판매일  품목코드        품목명 수량    단가       금액
BS-1102-001 2026.11.02 P-401 건전지 AA 20개 15  9900 148,500원
BS-1102-002 2026.11.02 P-301  믹스커피 10

수원점 주간 판매 현황 (11/02~11/08)                 NaN   NaN     NaN NaN  NaN    NaN
                       NaN                 NaN   NaN     NaN NaN  NaN    NaN
                      전표번호                 판매일  품목코드     품목명  수량   단가     금액
               SW-1102-001 2026-11-02 00:00:00 P-103 파일철 10개  20 9200 184000



## 2. 열 이름이 적힌 줄 찾기
수원점은 셋째 줄에 열 이름이 있다. 줄 번호를 외워 두지 않고, 아는 열 이름이 네 개 이상 있는 줄을 찾는다.

In [3]:
# 보고서가 쓰는 표준 열 이름
STANDARD = ["전표번호", "판매일", "품목코드", "품목명", "수량", "단가", "금액"]

# 지점마다 다르게 쓰는 열 이름 → 표준 이름
COLUMN_NAMES = {
    "일자": "판매일",
    "코드": "품목코드",
    "상품명": "품목명",
    "판매수량": "수량",
    "판매단가": "단가",
    "판매금액": "금액",
}

In [4]:
def find_header_row(path, max_rows=10):
    """앞쪽 몇 줄을 읽어, 열 이름이 적힌 줄이 몇 번째인지(0부터) 찾는다."""
    known = set(STANDARD) | set(COLUMN_NAMES)
    top = pd.read_excel(path, header=None, nrows=max_rows)
    for i, row in top.iterrows():
        cells = {str(v).strip() for v in row.tolist()}
        if len(cells & known) >= 4:
            return i
    raise ValueError(
        f"{path.name}: 열 이름이 적힌 줄을 찾지 못했습니다. "
        f"위쪽 {max_rows}줄 안에 {STANDARD} 같은 이름이 있어야 합니다."
    )


for f in files:
    print(f.name, "→", find_header_row(f))

강남점_주간판매.xlsx → 0
광주점_주간판매.xlsx → 0
대전점_주간판매.xlsx → 0
부산점_주간판매.xlsx → 0
수원점_주간판매.xlsx → 2


## 3. 파일 하나를 표준 모양으로 읽는 함수
열 이름을 맞추고, 필요한 열이 없으면 어느 파일의 무슨 열인지 알려 주고 멈춘다.
지점 이름은 파일 이름 앞부분에서 가져온다.

In [5]:
def read_branch_file(path):
    """지점 파일 하나를 읽어 표준 열 이름으로 맞춘다."""
    path = Path(path)
    df = pd.read_excel(path, header=find_header_row(path))
    df.columns = [str(c).strip() for c in df.columns]
    df = df.rename(columns=COLUMN_NAMES)

    missing = [c for c in STANDARD if c not in df.columns]
    if missing:
        raise ValueError(f"{path.name}: 필요한 열이 없습니다 → {missing}")

    df = df[STANDARD].copy()
    # 맥에서 만든 파일 이름은 자모가 풀려 있을 수 있다
    name = unicodedata.normalize("NFC", path.stem)
    df.insert(0, "지점", name.split("_")[0])
    df.insert(1, "원본파일", name + path.suffix)
    return df


read_branch_file(files[2]).head(3)

,지점,원본파일,전표번호,판매일,품목코드,품목명,수량,단가,금액
0,대전점,대전점_주간판매.xlsx,DJ-1102-001,2026-11-02,P-203,주방세제 4개,15,8700,130500
1,대전점,대전점_주간판매.xlsx,DJ-1102-002,2026-11-02,P-201,물티슈 10팩,24,11900,285600
2,대전점,대전점_주간판매.xlsx,DJ-1102-003,2026-11-02,P-402,멀티탭 4구,3,12500,37500


## 4. 다섯 개를 이어 붙이기

In [6]:
merged = pd.concat([read_branch_file(f) for f in files], ignore_index=True)

print("합친 줄 수:", len(merged))
merged.groupby("지점").size()

합친 줄 수: 259


지점
강남점    68
광주점    49
대전점    40
부산점    51
수원점    51
dtype: int64

줄 수가 맞는지 엑셀에서 센 것과 견준다. 아래는 파일마다 열 이름 줄 아래에 있는 줄 수를 따로 센 것이다.
두 숫자가 같아야 한 줄도 빠지지 않은 것이다.

In [7]:
for f in files:
    raw = pd.read_excel(f, header=None)
    print(f.name, len(raw) - find_header_row(f) - 1)

강남점_주간판매.xlsx 68
광주점_주간판매.xlsx 49


대전점_주간판매.xlsx 40


부산점_주간판매.xlsx 51
수원점_주간판매.xlsx 51


## 5. 중간 결과 저장
`utf-8-sig`로 저장해야 엑셀에서 열었을 때 한글이 깨지지 않는다.

In [8]:
WORK.mkdir(exist_ok=True)
merged.to_csv(WORK / "02_merged.csv", index=False, encoding="utf-8-sig")
merged.tail(3)

,지점,원본파일,전표번호,판매일,품목코드,품목명,수량,단가,금액
256,수원점,수원점_주간판매.xlsx,SW-1107-049,2026-11-07 00:00:00,P-102,유성펜 12자루,2.0,6800.0,13600
257,수원점,수원점_주간판매.xlsx,SW-1107-050,2026-11-07 00:00:00,P-402,멀티탭 4구,6.0,12500.0,75000
258,수원점,수원점_주간판매.xlsx,합계,NaT,NaN,NaN,515.0,NaN,5975100


## 실습. 지난주 폴더도 같은 함수로 합치기
폴더 이름만 바꾸면 된다. 주 하나를 통째로 읽는 함수로 묶어 둔다.

In [9]:
def read_week(week, data_dir=DATA):
    """raw/<week> 폴더의 엑셀 파일을 모두 읽어 한 표로 합친다."""
    week_dir = Path(data_dir) / "raw" / week
    if not week_dir.is_dir():
        raise FileNotFoundError(
            f"raw/{week} 폴더가 없습니다. "
            "지점 파일을 이 폴더에 넣었는지 확인하세요."
        )
    # '~$'로 시작하는 파일은 엑셀이 열려 있을 때 생기는 임시 파일이다
    files = sorted(p for p in week_dir.glob("*.xlsx") if not p.name.startswith("~$"))
    if not files:
        raise FileNotFoundError(f"raw/{week} 폴더에 엑셀 파일(.xlsx)이 없습니다.")
    return pd.concat([read_branch_file(f) for f in files], ignore_index=True)


prev = read_week("2026-10-26")
prev.to_csv(WORK / "02_merged_prev.csv", index=False, encoding="utf-8-sig")
print("지난주 줄 수:", len(prev))

지난주 줄 수: 258


없는 주를 넣으면 무엇이 문제인지 말해 주고 멈춘다.

In [10]:
try:
    read_week("2026-11-09")
except FileNotFoundError as error:
    print(error)

raw/2026-11-09 폴더가 없습니다. 지점 파일을 이 폴더에 넣었는지 확인하세요.
